# GADES Surface Analysis — Steps 1–3

This notebook implements the first three steps of the GADES post-processing workflow
on the `(x, U, F, H)` data logged during a GADES run.

| Step | What it does |
|------|--------------|
| **1 — Preprocessing** | Projects rigid-body modes out of each Hessian, diagonalises, computes per-snapshot scalar invariants |
| **2 — Feature construction** | Assembles the Z-scored feature matrix Φ ∈ ℝ^{N × (2K+5)} |
| **3 — Clustering** | HDBSCAN on Φ + mode-coherence check; labels each snapshot as basin, saddle candidate, or noise |

**Required log files** (produced by `GADESBias` when `logfile_prefix` is set):
```
<LOG_PREFIX>_pos.log
<LOG_PREFIX>_epot.log
<LOG_PREFIX>_forces.log
<LOG_PREFIX>_hess.log
```
All four files are expected in the same directory as this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import ListedColormap
import warnings

import GADES.SurfAnalysis.preprocessing as step1_mod
import GADES.SurfAnalysis.features     as step2_mod
import GADES.SurfAnalysis.clustering   as step3_mod

## Configuration

Edit the variables in this cell to match your run.

In [ ]:
# Path prefix — the logfile_prefix you passed to GADESBias.
# E.g. if your files are named  ./gades_pos.log  set LOG_PREFIX = "gades".
LOG_PREFIX = "gades"

# Simulation temperature in Kelvin (used for quasiharmonic free energy A_i).
TEMPERATURE = 300.0

# Boltzmann constant matching the backend that produced the run:
#   OpenMM  →  8.314462618e-3  kJ mol⁻¹ K⁻¹  (default)
#   ASE     →  8.617333262e-5  eV K⁻¹
KB = 8.314462618e-3

# Atomic masses of the biased atoms (same order as positions in _pos.log).
# Set to None to use uniform masses for the rigid-body projector.
MASSES = None

# Number of soft-mode eigenvectors to retain from Step 1.
N_EIGVECS = 20

# Number of eigenvalue modes K to include in the feature vector (Step 2).
# None → use all non-rigid modes.
K_FEATURES = None

# HDBSCAN parameters (Step 3).
HDBSCAN_MIN_CLUSTER_SIZE = 20
HDBSCAN_MIN_SAMPLES      = 5
COHERENCE_THRESHOLD      = 0.8

---
## Step 1 — Preprocessing and diagnostics

In [ ]:
s1 = step1_mod.run(
    logfile_prefix = LOG_PREFIX,
    temperature    = TEMPERATURE,
    masses         = MASSES,
    n_eigvecs      = N_EIGVECS,
    kB             = KB,
)

N = len(s1.steps)
print(f"Loaded {N} snapshots  |  n_rigid = {s1.n_rigid}  |  "
      f"n_nontrivial = {s1.eigenvalues.shape[1]}")

### Diagnostic plots

In [ ]:
t = np.arange(N)   # snapshot index

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
fig.suptitle("Step 1 — per-snapshot diagnostics", fontsize=13)

# ── panel 1: energy ──────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(t, s1.U, lw=0.8, label="$U_i$")
ax.plot(t, s1.A, lw=0.8, alpha=0.7, label="$A_i$")
ax.set_ylabel("Energy")
ax.legend(fontsize=9)

# ── panel 2: eigenvalues and spectral gap ────────────────────────────────────
ax = axes[1]
ax.plot(t, s1.eigenvalues[:, 0], lw=0.8, label=r"$\lambda_1$")
if s1.eigenvalues.shape[1] >= 2:
    ax.plot(t, s1.eigenvalues[:, 1], lw=0.8, label=r"$\lambda_2$")
    ax.plot(t, s1.spectral_gap,       lw=0.6, ls="--", label=r"$g = \lambda_2 - \lambda_1$")
ax.axhline(0, color="k", lw=0.5, ls=":")
ax.set_ylabel("Eigenvalue")
ax.legend(fontsize=9)

# ── panel 3: Morse index ─────────────────────────────────────────────────────
ax = axes[2]
ax.step(t, s1.morse_index, where="mid", lw=0.9, color="C2")
ax.set_ylabel("Morse index $m_i$")
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# ── panel 4: forces ──────────────────────────────────────────────────────────
ax = axes[3]
ax.plot(t, s1.F_norm,     lw=0.8, label="$|F_i|$")
ax.plot(t, s1.f_parallel, lw=0.8, label=r"$f_{\parallel}^{(i)}$")
ax.axhline(0, color="k", lw=0.5, ls=":")
ax.set_ylabel("Force")
ax.set_xlabel("Snapshot index")
ax.legend(fontsize=9)

for ax in axes:
    ax.grid(True, lw=0.3, alpha=0.5)

plt.tight_layout()
plt.savefig("step1_diagnostics.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Saddle-visit summary
n_saddle = int((s1.morse_index >= 1).sum())
print(f"Snapshots with m ≥ 1  (saddle-like) : {n_saddle:4d}  ({100*n_saddle/N:.1f}%)")
print(f"Snapshots with m = 0  (basin-like)  : {N - n_saddle:4d}  ({100*(N-n_saddle)/N:.1f}%)")
print(f"NaN A_i (no positive eigenvalues)   : {int(np.isnan(s1.A).sum()):4d}")

---
## Step 2 — Feature construction

In [ ]:
s2 = step2_mod.run(s1, K=K_FEATURES)

print(f"Feature matrix Φ : {s2.Phi.shape}  (K = {s2.K})")
print(f"Feature names     : {s2.feature_names}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Step 2 — feature matrix", fontsize=13)

# Raw feature matrix
im0 = axes[0].imshow(
    s2.Phi_raw.T, aspect="auto", interpolation="none",
    cmap="RdBu_r", origin="lower",
)
axes[0].set_title("Raw  Φ_raw")
axes[0].set_xlabel("Snapshot index")
axes[0].set_ylabel("Feature")
axes[0].set_yticks(range(len(s2.feature_names)))
axes[0].set_yticklabels(s2.feature_names, fontsize=7)
plt.colorbar(im0, ax=axes[0])

# Standardised feature matrix
vmax = np.percentile(np.abs(s2.Phi), 98)
im1 = axes[1].imshow(
    s2.Phi.T, aspect="auto", interpolation="none",
    cmap="RdBu_r", vmin=-vmax, vmax=vmax, origin="lower",
)
axes[1].set_title("Standardised  Φ")
axes[1].set_xlabel("Snapshot index")
axes[1].set_yticks(range(len(s2.feature_names)))
axes[1].set_yticklabels(s2.feature_names, fontsize=7)
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.savefig("step2_features.pdf", bbox_inches="tight")
plt.show()

---
## Step 3 — Clustering into landscape regions

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    s3 = step3_mod.run(
        s2, s1,
        min_cluster_size    = HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples         = HDBSCAN_MIN_SAMPLES,
        coherence_threshold = COHERENCE_THRESHOLD,
    )

for w in caught:
    print(f"[warning] {w.message}")

n_clusters = len(s3.cluster_ids)
print(f"Clusters found : {n_clusters}")
print(f"Noise points   : {s3.n_noise} / {N}  ({100*s3.n_noise/N:.1f}%)")

In [ ]:
# Per-cluster summary table
print(f"{'Cluster':>7}  {'Size':>5}  {'⟨U⟩':>10}  {'⟨λ₁⟩':>10}  {'⟨m⟩':>5}  {'sC':>5}  Type")
print("-" * 62)
for j, cid in enumerate(s3.cluster_ids):
    mean_m = s3.cluster_mean_morse[j]
    ctype  = "saddle" if mean_m >= 1.0 else ("near-saddle" if mean_m >= 0.3 else "basin")
    print(
        f"{cid:>7d}  "
        f"{s3.cluster_sizes[j]:>5d}  "
        f"{s3.cluster_mean_U[j]:>10.3f}  "
        f"{s3.cluster_mean_lambda1[j]:>10.3f}  "
        f"{mean_m:>5.2f}  "
        f"{s3.cluster_coherence[j]:>5.2f}  "
        f"{ctype}"
    )

In [ ]:
# Cluster labels over time
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
fig.suptitle("Step 3 — cluster assignment", fontsize=13)

# Colour map: grey for noise (-1), distinct colours for clusters
unique_labels = np.unique(s3.labels)
cmap_colors   = plt.get_cmap("tab20", max(n_clusters, 1))

label_colors = {}
cluster_idx  = 0
for lbl in sorted(unique_labels):
    if lbl == -1:
        label_colors[lbl] = (0.7, 0.7, 0.7, 1.0)  # grey noise
    else:
        label_colors[lbl] = cmap_colors(cluster_idx)
        cluster_idx += 1

point_colors = np.array([label_colors[l] for l in s3.labels])

# Panel 1: cluster label vs snapshot
axes[0].scatter(t, s3.labels, c=point_colors, s=4, linewidths=0)
axes[0].set_ylabel("Cluster label")
axes[0].yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# Panel 2: potential energy coloured by cluster
axes[1].scatter(t, s1.U, c=point_colors, s=4, linewidths=0)
axes[1].set_ylabel("$U_i$")
axes[1].set_xlabel("Snapshot index")

# Legend patches
from matplotlib.patches import Patch
handles = [Patch(color=label_colors[-1], label="noise (-1)")]
for j, cid in enumerate(s3.cluster_ids):
    mean_m = s3.cluster_mean_morse[j]
    ctype  = "saddle" if mean_m >= 1.0 else ("~saddle" if mean_m >= 0.3 else "basin")
    handles.append(Patch(color=label_colors[int(cid)], label=f"C{cid} ({ctype}, n={s3.cluster_sizes[j]})"))
axes[0].legend(handles=handles, fontsize=7, loc="upper right", ncol=2)

for ax in axes:
    ax.grid(True, lw=0.3, alpha=0.5)

plt.tight_layout()
plt.savefig("step3_clusters.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# 2-D PCA of Φ coloured by cluster label
from numpy.linalg import svd

Phi_c    = s2.Phi - s2.Phi.mean(axis=0)
_, _, Vt = svd(Phi_c, full_matrices=False)
PC       = Phi_c @ Vt[:2].T          # (N, 2)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(PC[:, 0], PC[:, 1], c=point_colors, s=8, linewidths=0, alpha=0.7)

# Mark cluster centroids
for j, cid in enumerate(s3.cluster_ids):
    mask = s3.labels == cid
    cx, cy = PC[mask, 0].mean(), PC[mask, 1].mean()
    ax.text(cx, cy, f"C{cid}", ha="center", va="center", fontsize=8,
            color="black", fontweight="bold")

ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_title("PCA of Φ — coloured by cluster")
ax.grid(True, lw=0.3, alpha=0.5)
ax.legend(handles=handles, fontsize=7, loc="best", ncol=2)

plt.tight_layout()
plt.savefig("step3_pca.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Mode-coherence bar chart
if n_clusters > 0:
    fig, ax = plt.subplots(figsize=(max(4, n_clusters * 0.6), 3))
    bars = ax.bar(
        range(n_clusters), s3.cluster_coherence,
        color=[label_colors[int(cid)] for cid in s3.cluster_ids],
        edgecolor="k", linewidth=0.5,
    )
    ax.axhline(COHERENCE_THRESHOLD, color="k", ls="--", lw=1,
               label=f"threshold = {COHERENCE_THRESHOLD}")
    ax.set_xticks(range(n_clusters))
    ax.set_xticklabels([f"C{cid}" for cid in s3.cluster_ids])
    ax.set_ylabel(r"Coherence score $s_C$")
    ax.set_ylim(0, 1.05)
    ax.set_title("Mode coherence per cluster")
    ax.legend()
    ax.grid(True, axis="y", lw=0.3, alpha=0.5)
    plt.tight_layout()
    plt.savefig("step3_coherence.pdf", bbox_inches="tight")
    plt.show()

---
## Saddle candidates

Clusters with mean Morse index ≥ 1, sorted by mean |F| ascending
(lowest force = closest to a true saddle).

In [ ]:
saddle_ids = s3.cluster_ids[s3.cluster_mean_morse >= 1.0]

if len(saddle_ids) == 0:
    print("No saddle-candidate clusters found (⟨m⟩ < 1 everywhere).")
else:
    print(f"{'Cluster':>7}  {'Size':>5}  {'⟨U⟩':>10}  {'⟨λ₁⟩':>10}  {'⟨m⟩':>5}  {'⟨|F|⟩':>9}  {'sC':>5}")
    print("-" * 60)
    # Sort by mean |F| within saddle clusters
    mean_F_per_cluster = {
        int(cid): s1.F_norm[s3.labels == cid].mean()
        for cid in saddle_ids
    }
    for cid in sorted(saddle_ids, key=lambda c: mean_F_per_cluster[int(c)]):
        j = np.where(s3.cluster_ids == cid)[0][0]
        print(
            f"{cid:>7d}  "
            f"{s3.cluster_sizes[j]:>5d}  "
            f"{s3.cluster_mean_U[j]:>10.3f}  "
            f"{s3.cluster_mean_lambda1[j]:>10.3f}  "
            f"{s3.cluster_mean_morse[j]:>5.2f}  "
            f"{mean_F_per_cluster[int(cid)]:>9.4f}  "
            f"{s3.cluster_coherence[j]:>5.2f}"
        )

---
## Export results

Save the step results to NumPy files for downstream use (Steps 4–6).

In [ ]:
np.save("eigenvalues.npy",  s1.eigenvalues)
np.save("eigenvectors.npy", s1.eigenvectors)
np.save("features.npy",     s2.Phi)
np.save("labels.npy",       s3.labels)

# Cluster summary as a structured array
import csv
with open("cluster_summary.csv", "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["cluster_id", "size", "mean_U", "mean_lambda1",
                     "mean_morse", "coherence"])
    for j, cid in enumerate(s3.cluster_ids):
        writer.writerow([
            int(cid),
            int(s3.cluster_sizes[j]),
            float(s3.cluster_mean_U[j]),
            float(s3.cluster_mean_lambda1[j]),
            float(s3.cluster_mean_morse[j]),
            float(s3.cluster_coherence[j]),
        ])

print("Saved: eigenvalues.npy, eigenvectors.npy, features.npy, labels.npy, cluster_summary.csv")